In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-03-20 03:33:23--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  6.19MB/s    in 0.2s    

2026-03-20 03:33:23 (6.19 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open('input.txt','r',encoding='utf-8') as file:
  text=file.read()

In [ ]:
print('length of the dataset :', len(text))

length of the dataset : 1115394


In [ ]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(vocab_size)

65


In [ ]:
soti ={ ch:i for i,ch in enumerate(chars)}
itos ={i:ch for i,ch in enumerate(chars)}
encode= lambda s: [soti[c] for c in s]
decode = lambda s: ''.join([itos[c] for c in s])

print(encode("hi hello"))
print(decode(encode("hi hello")))


[46, 47, 1, 46, 43, 50, 50, 53]
hi hello


In [ ]:
import torch

In [ ]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.type())

torch.Size([1115394]) torch.LongTensor


In [ ]:
n= int(0.9*len(data))
train_data=data[:n]
test_data=data[n:]

In [ ]:

batch_size=4
block_size=8
def get_batch(string):
  if string=='train':
    data=train_data
  else:
    data=test_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+1+block_size] for i in ix])
  return x,y

xb,yb=get_batch('train')
print('inputs')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)
print('============================================================================')
for b in range(batch_size):
  for c in range(block_size):
    context=xb[b,:c+1]
    target=yb[b,c]
    print(f'for this context {context.tolist()} the target is {target}')


inputs
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
for this context [24] the target is 43
for this context [24, 43] the target is 58
for this context [24, 43, 58] the target is 5
for this context [24, 43, 58, 5] the target is 57
for this context [24, 43, 58, 5, 57] the target is 1
for this context [24, 43, 58, 5, 57, 1] the target is 46
for this context [24, 43, 58, 5, 57, 1, 46] the target is 43
for this context [24, 43, 58, 5, 57, 1, 46, 43] the target is 39
for this context [44] the target is 53
for this context [44, 53] the target is 56
for this context [44, 53, 56] the target is 1
for this context [44, 53, 56, 1] the target is 58
for this context 

In [ ]:
#final script to run

import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

#define variables

n_embed=32
batch_size=8
block_size=4
max_iter=3000
eval_iter=200
eval_interval=300
learning_rate=1e-2

#data import
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt  # remove ! for script
with open('input.txt','r',encoding='utf-8') as file:
  text=file.read()
chars=sorted(list(set(text)))
vocab_size=len(chars)

#encoder and decoder
soti ={ ch:i for i,ch in enumerate(chars)}
itos ={i:ch for i,ch in enumerate(chars)}
encode= lambda s: [soti[c] for c in s]
decode = lambda s: ''.join([itos[c] for c in s])

#train data and test data split
data = torch.tensor(encode(text),dtype=torch.long)
n= int(0.9*len(data))
train_data=data[:n]
test_data=data[n:]

def get_batch(string):
  if string=='train':
    data=train_data
  else:
    data=test_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+1+block_size] for i in ix])
  return x,y

@torch.no_grad()
def estimate_loss():
  out={}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iter)
    for k in range(eval_iter):
      x,y=get_batch(split)
      logit,loss=model(x,y)

      losses[k]=loss.item()
    out[split]=losses.mean()
  model.train()
  return out


class biagramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embeding_table=nn.Embedding(vocab_size,n_embed)
    self.pos_embedding_table=nn.Embedding(block_size,n_embed)
    self.lm__head=nn.Linear(n_embed,vocab_size)

  def forward(self,input,targets=None):
    B,T=idx.shape
    token_emb=self.token_embeding_table(input)  # returns of shape (B,T,n_embed)
    pos_emb=self.pos_embedding_table(torch.arange(T)) #returns (T,n_embed)
    x=token_emb+pos_emb
    logits=self.lm__head(x)

    if targets==None:
      loss=None
    else:
      b,t,c=logits.shape
      logits=logits.view(b*t,c)
      targets=targets.view(b*t)
      loss=F.cross_entropy(logits,targets)

    return logits,loss

  def generate(self, idx, max_new_tokens):
    for i in range(max_new_tokens):
      logits,loss= self(idx) #returns logits in shape
      logits=logits[:,-1,:] #returns only (b,c)
      prob=F.softmax(logits,dim=1)
      next_idx=torch.multinomial(prob,num_samples=1)# produce only next xharaxter from the prob
      idx=torch.concat((idx,next_idx),dim=1)
    return idx

model=biagramLanguageModel(vocab_size)
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)

for iter in range(max_iter):
  if iter % eval_interval == 0:
    losses=estimate_loss()
    print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
  xb,yb=get_batch('train')

  logits,loss=model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())


idx=torch.zeros((1,1),dtype=torch.long)
print(decode(model.generate(idx,100)[0].tolist()))

torch.Size([32, 65]) tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
#create the optimier
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)

In [ ]:
batch_size=32
for i in range(10000):
  xb,yb=get_batch(train_data)
  logits,loss=model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.2970616817474365


In [ ]:
idx=torch.zeros((1,1),dtype=torch.long)
print(decode(model.generate(idx,100)[0].tolist()))


HIO:
b$

Binis t bean for e,


Pefaniss:
Ther PRIO: ds ar,
Whe, wnche owathore; totitous twh anend t
